In [1]:
hidden_dim = 256
emb_dim = 16
vocab_size = 50

In [2]:
import torch
import torch.nn as nn

class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CharLSTM, self).__init__()
        # 1. The Dictionary
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        # 2. The Memory (batch_first=True)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        # 3. The Output (Maps hidden state back to vocab size)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x, states):
        # x: (batch, seq_len) -> integers
        out = self.embedding(x) # out: (batch, seq_len, embed_dim)
        out, states = self.lstm(out, states)
        # Flatten for the linear layer
        out = out.reshape(-1, out.shape[2]) 
        out = self.fc(out)
        return out, states

In [3]:
# --- THE CHALLENGE ---

def generate_step(model, char_idx, last_states):
    """
    Predicts the next character based on ONE input character.
    """
    # 1. Prepare the input: Wrap char_idx in a tensor with shape (1, 1) 
    # (Batch size 1, Sequence length 1)
    input_tensor = torch.tensor([[char_idx]])
    
    # 2. Pass the input and last_states through the model
    # Hint: model(input, states) returns (output, new_states)
    logits, new_states = model(input_tensor, last_states)
    
    # 3. Get the "scores" for the characters (the last dimension)
    # Since batch and seq are 1, we can just flatten or squeeze
    probs = torch.softmax(logits.squeeze(), dim=0)
    
    # 4. Pick the character with the highest probability
    next_char_idx = torch.argmax(probs)
    
    return next_char_idx.item(), new_states

In [4]:
# Test your logic
vocab_size, embed_dim, hidden_dim = 50, 16, 256
model = CharLSTM(vocab_size, embed_dim, hidden_dim)

# Initialize states to None (LSTM handles zero-init if None is passed)
current_char = 10 # Let's say 'a' is index 10
states = None

# Try to generate 1 step
next_char, states = generate_step(model, current_char, states)
print(f"Predicted next char index: {next_char}")

Predicted next char index: 39
